# Synthetic Data Generation with Local LLaMA Model

This tutorial demonstrates how to use SDG repository to generate synthetic question-answer pairs from documents using a local LLaMA model running through vLLM. We'll cover:

1. Setting up the environment
2. Connecting to local vLLM server
3. Configuring the data generation pipeline
4. Generating synthetic data

In [1]:
# Enable auto-reloading of modules - useful during development
%load_ext autoreload
%autoreload 2

### Setup Instructions

Before running this notebook, ensure SDG Hub is installed:

```python
import sys
sys.path.insert(0, '/home/akamra/sdg_hub/src') # replace with the path to your sdg_hub/src location
```

Or install it directly:
```bash 
pip install -e /home/akamra/sdg_hub #replace with the path to your sdg_hub/src location
```

In [1]:
# Import required libraries
import sys
sys.path.insert(0, '/home/akamra/sdg_hub/src') #replace with the path to your sdg_hub/src location

from datasets import load_dataset, Dataset
from openai import OpenAI

from sdg_hub.flow import Flow
from sdg_hub.sdg import SDG

### Setting up Local LLaMA Model

Make sure your local vLLM server is running at http://localhost:8001/v1 with the model:
`model/RedHatAI/Meta-Llama-3.1-8B-Instruct-quantized.w4a16`

Example vLLM startup command using podman:
```bash
podman run --rm --name vllm-llama --device nvidia.com/gpu=all --ipc=host -p 8001:8001 -v ~/models/:/model:ro -v ~/.cache/huggingface/:/root/.cache/huggingface vllm-openai:v0.9.0 --model /model/RedHatAI/Meta-Llama-3.1-8B-Instruct-quantized.w4a16 --gpu-memory-utilization 0.9 --dtype auto --max_model_len 4096 --port 8001
```

In [2]:
# Configure OpenAI client to connect to our local vLLM server
endpoint = "http://localhost:8001/v1"
openai_api_key = "EMPTY"  # vLLM doesn't require real API key

client = OpenAI(
    api_key=openai_api_key,
    base_url=endpoint,
)

# Verify we can see the model
try:
    models = client.models.list()
    teacher_model = models.data[0].id
    print(f"Connected to model: {teacher_model}")
except Exception as e:
    print(f"Error connecting to model: {e}")
    print("Make sure your vLLM server is running at http://localhost:8001/v1")

[15:25:46] INFO     HTTP Request: GET http://localhost:8001/v1/models "HTTP/1.1 200 OK"             ]8;id=219034;file:///home/akamra/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=298334;file:///home/akamra/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/httpx/_client.py#1026\1026]8;;\

Connected to model: /model/RedHatAI/Meta-Llama-3.1-8B-Instruct-quantized.w4a16


### Configure the Data Generation Pipeline

Now we'll set up our Synthetic Data Generation (SDG) pipeline using the local model configuration.

In [3]:
# Load the flow configuration from YAML file for local model
flow_cfg = Flow(client).get_flow_from_file("synth_knowledge1.5_local_llama.yaml")

# Initialize the SDG pipeline with processing parameters
sdg = SDG(
    [flow_cfg],        # Use Flow directly, not wrapped in Pipeline
    num_workers=1,     # Number of parallel workers
    batch_size=1,      # Batch size for processing
    save_freq=1000,    # How often to save checkpoints
)

### Load and Prepare Seed Data

We'll create some sample data to demonstrate the data generation process.

In [4]:
# Create sample seed data
sample_document = """
Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines 
that can perform tasks that typically require human intelligence. These tasks include learning, reasoning, 
problem-solving, perception, and language understanding. AI systems can be categorized into two main types: 
narrow AI, which is designed for specific tasks, and general AI, which would have human-like cognitive abilities 
across multiple domains. Machine learning, a subset of AI, enables computers to learn and improve from experience 
without being explicitly programmed for every task.
"""

# Create a dataset from the sample document
data = [{"document": sample_document.strip()}]
ds = Dataset.from_list(data)

print(f"Created dataset with {len(ds)} document(s)")
print("Sample document (first 200 chars):", ds[0]['document'][:200] + "...")

Created dataset with 1 document(s)
Sample document (first 200 chars): Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines 
that can perform tasks that typically require human intelligence. These tasks include learning, r...


### Generate Synthetic Data

Now we'll use our configured pipeline to generate synthetic question-answer pairs from the document.

In [5]:
# Generate synthetic data and save checkpoints
print("Starting data generation...")
try:
    generated_data = sdg.generate(ds, checkpoint_dir="local_llama_checkpoint")
    print(f"Successfully generated {len(generated_data)} examples")
except Exception as e:
    print(f"Error during generation: {e}")
    import traceback
    traceback.print_exc()

Starting data generation...


[15:25:53] INFO     No existing checkpoints found in local_llama_checkpoint, generating from     ]8;id=841369;file:///home/akamra/sdg_hub/src/sdg_hub/checkpointer.py\checkpointer.py]8;;\:]8;id=534559;file:///home/akamra/sdg_hub/src/sdg_hub/checkpointer.py#72\72]8;;\
                    scratch                                                                                        

           INFO     Splitting the dataset into smaller batches                                           ]8;id=446827;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py\sdg.py]8;;\:]8;id=507965;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py#167\167]8;;\

100%|██████████| 1/1 [00:00<00:00, 38130.04it/s]


           INFO     Generating dataset with 1 splits, batch size 1, and 1 workers                        ]8;id=348058;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py\sdg.py]8;;\:]8;id=406396;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py#169\169]8;;\

...

Filter:   0%|          | 0/28 [00:00<?, ? examples/s]

Filter:   0%|          | 0/28 [00:00<?, ? examples/s]

           INFO     Finished future processing split 0                                                   ]8;id=524135;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py\sdg.py]8;;\:]8;id=442820;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py#190\190]8;;\
                                                                                                                   
                                                                                                                   

100%|██████████| 1/1 [01:00<00:00, 60.13s/it]

Successfully generated 28 examples


### Examine Generated Results

Let's look at the generated question-answer pairs.

In [6]:
# Display the generated results
if 'generated_data' in locals() and generated_data is not None:
    print("Generated Data Sample:")
    print("=" * 50)
    
    for i, example in enumerate(generated_data):
        if i >= 3:  # Show first 3 examples
            break
            
        print(f"\nExample {i+1}:")
        print(f"Question: {example.get('question', 'N/A')}")
        print(f"Response: {example.get('response', 'N/A')}")
        print("-" * 30)
    
    # Show column names
    print(f"\nDataset columns: {generated_data.column_names}")
    print(f"Total examples generated: {len(generated_data)}")
else:
    print("No data was generated. Check the error messages above.")

Generated Data Sample:

Example 1:
Question: What are the primary characteristics of Artificial Intelligence (AI) systems?
Response: Artificial Intelligence (AI) systems are characterized by their ability to perform tasks that typically require human intelligence, including learning, reasoning, problem-solving, perception, and language understanding. 
------------------------------

Example 2:
Question: What is the primary difference between Narrow AI and General AI systems?
Response: The primary difference between Narrow AI and General AI systems is that Narrow AI systems are designed for specific tasks and are limited to a particular domain or function, whereas General AI systems would possess human-like cognitive abilities across multiple domains, enabling them to perform a wide range of tasks. 
------------------------------

Example 3:
Question: What is Machine Learning, and how does it contribute to AI systems?
Response: Machine Learning is a subset of AI that enables computers t

### Save Results

Save the generated data to a file for later use.

In [7]:
# Save the generated data
if 'generated_data' in locals() and generated_data is not None:
    output_file = "generated_local_llama_data.jsonl"
    generated_data.to_json(output_file)
    print(f"Saved generated data to {output_file}")
    
    # Also save as CSV for easier viewing
    csv_file = "generated_local_llama_data.csv"
    generated_data.to_csv(csv_file)
    print(f"Saved generated data to {csv_file}")
else:
    print("No data to save.")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved generated data to generated_local_llama_data.jsonl


Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved generated data to generated_local_llama_data.csv


### Test Different Documents

You can also test with your own documents by replacing the sample_document variable above or loading from a file.

In [ ]:
# Example: Load documents from a JSON file
# Uncomment and modify the path below to use your own data

# your_data_path = "path/to/your/documents.json"
# ds_custom = load_dataset('json', data_files=your_data_path, split='train')
# generated_custom = sdg.generate(ds_custom.select(range(1)), checkpoint_dir="custom_checkpoint")
# print(f"Generated {len(generated_custom)} examples from custom data")